In [1]:
import requests
import torch
import time
import psutil
import subprocess

import pandas as pd

from datasets import load_dataset

In [2]:
ds = load_dataset("KushT/bbc_news_multiclass_train_val_test")

test = ds['test'].to_pandas()

test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 334 entries, 0 to 333
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    334 non-null    object
 1   label   334 non-null    int64 
dtypes: int64(1), object(1)
memory usage: 5.3+ KB


In [3]:
test['label'] = test['label'].apply(lambda x: 'business' if x == 0 else 'entertainment' if x == 1 else 'politics' if x == 2 else 'sport' if x == 3 else 'tech')

labels = test['label'].unique()

test

,text,label
0,Dogged Federer claims Dubai crown World number...,sport
1,UK troops on Ivory Coast standby Downing Stree...,politics
2,Keanu Reeves given Hollywood star Actor Keanu ...,entertainment
3,Classy Henman makes winning start Tim Henman o...,sport
4,Mixed reaction to Man Utd offer Shares in Manc...,business
...,...,...
329,Middlesbrough 2-2 Charlton A late header by te...,sport
330,Budget Aston takes on Porsche British car make...,business
331,Hi-tech posters guide commuters Interactive po...,tech
332,Hotspot users gain free net calls People using...,tech


In [4]:
def get_ollama_memory_usage(port=11434):
    """
    Finds the process listening on the given port using psutil
    and returns its memory usage in bytes (RSS).
    Returns None if the process isn't found or can't be accessed.
    """
    for proc in psutil.process_iter(['pid', 'name']):
        try:
            # Call proc.connections() to see if it's listening on the desired port
            for conn in proc.connections(kind='inet'):
                if conn.laddr.port == port:
                    # Found the process that listens on port=11434
                    memory_info = proc.memory_info()
                    return memory_info.rss  # in bytes
        except (psutil.AccessDenied, psutil.NoSuchProcess):
            pass
    
    # If no process was found
    return None

get_ollama_memory_usage()

C:\Users\Rafael\AppData\Local\Temp\ipykernel_22232\1313842216.py:10: DeprecationWarning: connections() is deprecated and will be removed; use net_connections() instead
  for conn in proc.connections(kind='inet'):


53977088

In [5]:
def get_gpu_memory_usage():
    """
    Returns a list of used memory (in MB) for each GPU.
    """
    # Use nvidia-smi with the --query-gpu and --format flags to get just the memory usage
    command = [
        "nvidia-smi",
        "--query-gpu=memory.used",  # You can also add memory.free, name, etc.
        "--format=csv,noheader,nounits"  # CSV output with no header or units
    ]
    try:
        output = subprocess.check_output(command)
        # Decode the output from bytes to string
        output_str = output.decode("utf-8").strip()
        # Each line corresponds to one GPU's memory usage
        usage_values = [int(x) for x in output_str.split("\n")]
        return usage_values[0]
    except subprocess.CalledProcessError as e:
        print("Error running nvidia-smi:", e)
        return []


In [ ]:
def classify(text, labels):

    url = "http://localhost:11434/api/chat"

    payload = {
        "model": "llama3.2:3b",
        "messages" : [
            {"role": "system", "content": "You are a classification assistant. Your objective is to read the provided text and classify it according to the task and labels described. You are capable of handling multiclass classification tasks based on user instructions."},
            {"role": "user", "content": f"Classify the following text based on the task: Category classification of news articles. Only respond with the label that best describe the text. The possible labels are: {', '.join(labels)}. Text: {text}"}
        ],
        "stream": False,
        "options": {
            "temperature": 0
        }
    }

    start_time = time.time()
    response = requests.post(url, json=payload)
    response_time = time.time() - start_time

    vram_usage = get_gpu_memory_usage()

    ram_usage_bytes = get_ollama_memory_usage(port=11434) / (1024 * 1024)

    response = response.json()
    total_time = response['total_duration'] / 1_000_000_000
    content = response['message']['content'].lower()

    if 'business' in content:
        content = 'business'
    elif 'sport' in content:
        content = 'sport'
    elif 'entertainment' in content:
        content = 'entertainment'
    elif 'politics' in content:
        content = 'politics'
    elif 'tech' in content:
        content = 'tech'
    else:
        content = 'error'

    return content, response_time, vram_usage, ram_usage_bytes, total_time

In [8]:
# apply the classify function to the test set. create one column for each output
test[['prediction', 'response_time', 'vram_usage', 'ram_usage', 'total_time']] = test['text'].apply(lambda x: classify(x, labels)).apply(pd.Series)

C:\Users\Rafael\AppData\Local\Temp\ipykernel_22232\1313842216.py:10: DeprecationWarning: connections() is deprecated and will be removed; use net_connections() instead
  for conn in proc.connections(kind='inet'):


business
business
sport
business
business
business
business
business
business
business
business
business
sport
business
business
business
business
business
business
business
business
business
business
business
entertainment
business
business
business
business
business
business
business
business
business
business
business
business
business
business
business
business
business
business
business
business
business
business
business
business
entertainment
entertainment
entertainment
entertainment
entertainment
entertainment
entertainment
entertainment
entertainment
entertainment
entertainment
business
entertainment
error
entertainment
entertainment
entertainment
business
entertainment
entertainment
entertainment
entertainment
entertainment
entertainment
entertainment
entertainment
entertainment
entertainment
entertainment
entertainment
entertainment
entertainment
entertainment
entertainment
entertainment
business
entertainment
entertainment
entertainment
entertainment
entertainment
entertain

In [9]:
test

,text,label,prediction,response_time,vram_usage,ram_usage,total_time
0,Gazprom 'in $36m back-tax claim' The nuclear u...,business,business,3.521067,4128,97.218750,1.499381
1,South African car demand surges Car manufactur...,business,business,2.135382,4107,97.441406,0.086088
2,Umbro profits lifted by Euro 2004 UK sportswea...,business,sport,2.156409,4107,98.035156,0.108766
3,Dollar hovers around record lows The US dollar...,business,business,2.436877,4107,98.097656,0.391634
4,Newest EU members underpin growth The European...,business,business,2.545339,4117,99.117188,0.512019
...,...,...,...,...,...,...,...
240,New browser wins over net surfers The proporti...,tech,business,2.529406,4158,98.855469,0.483634
241,How to make a gigapixel picture The largest di...,tech,entertainment,2.492715,4162,98.843750,0.444709
242,Mobile games come of age The BBC News website ...,tech,entertainment,2.752882,4168,98.906250,0.714408
243,Nintendo DS makes its Euro debut Nintendo's DS...,tech,entertainment,2.527372,4168,99.417969,0.482309


In [10]:
y_pred = test['prediction']
y_true = test['label']

#import acc, f1_score, precision and recall from sklearn
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

#calculate the accuracy of the model
accuracy = accuracy_score(y_true, y_pred)
print('Accuracy: %f' % accuracy)
f1 = f1_score(y_true, y_pred, average='weighted')
print('F1 score: %f' % f1)
precision = precision_score(y_true, y_pred, average='weighted')
print('Precision: %f' % precision)
recall = recall_score(y_true, y_pred, average='weighted')
print('Recall: %f' % recall)

Accuracy: 0.587755
F1 score: 0.520860
Precision: 0.795423
Recall: 0.587755


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [11]:
# get average response time, vram usage and ram usage
response_time_avg = test['response_time'].mean()
vram_usage_avg = test['vram_usage'].mean()
ram_usage_avg = test['ram_usage'].mean()
total_time_avg = test['total_time'].mean()

print(f'Average response time: {response_time_avg}')
print(f'Average VRAM usage: {vram_usage_avg}')
print(f'Average RAM usage: {ram_usage_avg}')
print(f'Average total time: {total_time_avg}')

Average response time: 2.549450804262745
Average VRAM usage: 4126.902040816327
Average RAM usage: 99.06141581632653
Average total time: 0.505085698367347


In [12]:
# save results to txt
with open('results/gemma_ZS_multiclass2.txt', 'w') as f:
    f.write(f'Accuracy: {accuracy}\n')
    f.write(f'F1 score: {f1}\n')
    f.write(f'Precision: {precision}\n')
    f.write(f'Recall: {recall}\n')
    f.write(f'Average response time: {response_time_avg}\n')
    f.write(f'Average VRAM usage: {vram_usage_avg}\n')
    f.write(f'Average RAM usage: {ram_usage_avg}\n')
    f.write(f'Average total time: {total_time_avg}\n')